# Problem

**State space.**
The states are encoded as integers
$$\mathcal{X} = {0,1,\dots,8} $$
corresponding to the grid in row-major order (top-left corner to bottom-right corner).
- State $8$ is the terminal goal state and is absorbing.
- State $5$ is the pit state and is absorbing.
- State $4$ is a wall.


**Action space.**
Actions are represented as integers

$$ \mathcal{A} = {0,1,2,3} $$

where $0 =$ Up, $1 =$ Down, $2 =$ Left, $3 =$ Right.

**Transition dynamics.**
Transitions are deterministic. For any state $x$ and action $a$, the state index is first mapped to its grid coordinates $(r,c)$, the action moves the agent one step (unless this would leave the grid), and the result is mapped back to a state index. The goal state $8$ always transitions to itself.
The transition matrix $P$ has shape $(|\mathcal{X}||\mathcal{A}|) \times |\mathcal{X}| = 36 \times 9$, where each row corresponds to a pair $(x,a)$ and contains a one-hot vector indicating the unique next state:

$$P[(x,a),x'] = 1 \quad \text{iff } x' = \text{next\_state}(x,a)$$

and $0$ otherwise.

**Reward model.**
- $r(x,a) = 1$ if x is a goal state.
- $r(x,a) = -1$ if x is the pit state.
- $r(x,a) = -0.01$ otherwise.

**Initial state.**
The initial state is fixed as $x_0 = 0$.

In [15]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd
import sys
import torch
from pathlib import Path

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_root(current_path, marker="setup.py"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    return current_path

PROJECT_ROOT = find_root(Path.cwd())
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
ASSETS_DIR = PROJECT_ROOT / "experiments" / "shared" / "assets"
# Add project root to the Python path
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rl_methods import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Problem

In [16]:
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)
N = len(states) # number of states
A = len(actions) # number of actions
gamma = 0.9

x_0 = 0 # fixed initial state
dataset_path = str(DATASETS_DIR / "3gridw_tab_gen.csv") # path where to download the dataset

goal = 8   # absorbing terminal state
pit = 5   # absorbing terminal state

def phi(x, a):
    vec = torch.zeros(N * A, dtype=torch.float64)
    vec[int(x) * A + int(a)] = 1.0
    return vec

step_cost = -0.01
goal_reward = 1.0
pit_reward  = -1.0

omega = torch.full((N * A,), step_cost, dtype=torch.float64)

# terminals: you can either keep step cost or override; usually override
omega[goal * A : goal * A + A] = goal_reward
omega[pit  * A : pit  * A + A] = pit_reward


# Helper to convert index <-> (row, col)
def to_rc(s):  return divmod(int(s), 3)
def to_s(r,c): return r*3 + c

wall = 4

def next_state(s, a):
    s = int(s)
    a = int(a)

    # absorbing terminals
    if s == goal or s == pit:
        return s

    r, c = to_rc(s)

    if a == 0:      # Up
        r2, c2 = max(0, r-1), c
    elif a == 1:    # Down
        r2, c2 = min(2, r+1), c
    elif a == 2:    # Left
        r2, c2 = r, max(0, c-1)
    elif a == 3:    # Right
        r2, c2 = r, min(2, c+1)

    sp = to_s(r2, c2)

    # wall blocks transition
    if sp == wall:
        return s

    return sp

def psi(xp):
    v = torch.zeros(N * A, dtype=torch.float64)
    for x in states:
        for a in actions:
            if next_state(x, a) == int(xp):
                v[int(x) * A + int(a)] = 1.0
    return v

mdp = PolicySolver(states=states, actions=actions, phi=phi, omega=omega, gamma=gamma, x0=x_0, psi=psi)


# Dataset Collection

In [9]:
collector = EnvDataCollector(
    mdp=mdp,
    max_steps=30,
    terminal_states=[goal],
    seed=42,
)

df = collector.collect_mixed_dataset(
    policies=[
        (mdp.pi_star, 0.3),   
        "random",
    ],
    proportions=[0.7, 0.3],
    n_steps=1000,
    save_path=dataset_path,
    verbose=True,
    episode_based=True,
)


  MIXED DATASET COLLECTION SUMMARY (TORCH)
Total transitions: 1000
Total episodes: 93
Mode: Episode-based

Policy Distribution:
  Policy 0:   432 steps (43.2%) | Target: 70.0% | Episodes: 72
  Policy 1:   568 steps (56.8%) | Target: 30.0% | Episodes: 22

✅ Mixed dataset saved to: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/3gridw_tab_gen.csv


# Empirical

## Beta

In [5]:
solver_e = VBetaSolver(mdp=mdp, print_params=True, csv_path=dataset_path, T=1000, seed=SEED, device=DEVICE)
evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))


Device: cpu
Dataset: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/data/datasets/3gridw_tab_gen.csv (n=1000)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           1000
Feature norm bound R:     1.0000
Num states N:             9
Num actions A:            4
Feature dim d:            36
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      925.5128526390366
T (iterations):                1000   (overridden → 1000)

FOGAS Hyperparameters
---------------------
alpha:                        0.000878
rho:                            596.628481
eta:                            0.000017
D_theta:                    18.973666
beta (ridge):             0.000028
D_pi (derived):           16.651092




In [7]:
solver_e.run(alpha=1e-3, eta=2e-4, rho=0.5, T = 2000, tqdm_print=True)
evaluator_e.compare_value_functions()
evaluator_e.compare_final_rewards()
evaluator_e.print_policy()

FOGAS:   0%|          | 0/2000 [00:00<?, ?it/s]

FOGAS: 100%|██████████| 2000/2000 [00:00<00:00, 2137.43it/s]


========== VALUE FUNCTION COMPARISON ==========

State-wise V comparison:
State 0: V*(x) =  6.526610 | V^π(x) =  1.257598 | Δ = -5.269012e+00
State 1: V*(x) =  5.863949 | V^π(x) = -5.824322 | Δ = -1.168827e+01
State 2: V*(x) =  5.267554 | V^π(x) = -7.895416 | Δ = -1.316297e+01
State 3: V*(x) =  7.262900 | V^π(x) =  5.536337 | Δ = -1.726563e+00
State 4: V*(x) =  8.081000 | V^π(x) = -0.495994 | Δ = -8.576994e+00
State 5: V*(x) = -10.000000 | V^π(x) = -10.000000 | Δ =  2.309264e-14
State 6: V*(x) =  8.081000 | V^π(x) =  6.803479 | Δ = -1.277521e+00
State 7: V*(x) =  8.990000 | V^π(x) =  8.128010 | Δ = -8.619903e-01
State 8: V*(x) =  10.000000 | V^π(x) =  10.000000 | Δ =  0.000000e+00

Action-value Q comparison:
(x=0, a=0): Q*(x,a) =  5.863949 | Q^π(x,a) =  1.121838 | Δ = -4.742111e+00
(x=0, a=1): Q*(x,a) =  6.526610 | Q^π(x,a) =  4.972704 | Δ = -1.553906e+00
(x=0, a=2): Q*(x,a) =  5.863949 | Q^π(x,a) =  1.121838 | Δ = -4.742111e+00
(x=0, a=3): Q*(x,a) =  5.267554 | Q^π(x,a) = -5.251890 |

### Objective update

In [10]:
solver_e = VBetaObjectivePolicySolver(
    mdp=mdp,
    csv_path=dataset_path,
    beta=1e-2,          # ridge
    print_params=True,  
    seed=SEED,
    device=DEVICE,
)

evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))


Device: cpu
Dataset: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/data/datasets/3gridw_tab_gen.csv (n=1000)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           1000
Feature norm bound R:     1.0000
Num states N:             9
Num actions A:            4
Feature dim d:            36
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      925.5128526390366
T (iterations):                926

FOGAS Hyperparameters
---------------------
alpha:                        0.000912
rho:                            594.460202
eta:                            0.000018
D_theta:                    18.973666
beta (ridge):             0.000030   (overridden → 0.010000)
D_pi (derived):           16.023162




In [ ]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
)
evaluator_e.print_policy()

[FOGAS objective-policy] Iter 1/2000 total_loss=-9.486833e-01 policy_loss=-9.486833e-01 theta_norm=1.897367e+01 beta_norm=3.395861e-03 grad_norm=1.698100e+01 policy_grad_norm=1.897367e+00 state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 state_weight_neg=0
[FOGAS objective-policy] Iter 51/2000 total_loss=-8.052905e-01 policy_loss=-9.268526e-01 theta_norm=1.897367e+01 beta_norm=1.558281e-01 grad_norm=1.377745e+01 policy_grad_norm=1.853917e+00 state_weight_min=-1.007000e-03 state_weight_max=9.893208e-02 state_weight_neg=2
[FOGAS objective-policy] Iter 101/2000 total_loss=-7.202632e-01 policy_loss=-9.051878e-01 theta_norm=1.897367e+01 beta_norm=2.763833e-01 grad_norm=1.131549e+01 policy_grad_norm=1.809791e+00 state_weight_min=-1.988142e-03 state_weight_max=9.971768e-02 state_weight_neg=1
[FOGAS objective-policy] Iter 151/2000 total_loss=-6.653952e-01 policy_loss=-8.938959e-01 theta_norm=1.897367e+01 beta_norm=3.733718e-01 grad_norm=9.856743e+00 policy_grad_norm=1.777179e+00 stat

Same thing! This new weights do that the policy update is not that strong in the not that well covered states.

In [17]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
)
evaluator_e.print_policy()

[FOGAS objective-policy] Iter 1/2000 total_loss=-9.486833e-01 policy_loss=-4.743416e+00 theta_norm=1.897367e+01 beta_norm=3.395861e-03 grad_norm=1.698100e+01 policy_grad_norm=9.486833e+00 state_weight_update=clipped state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS objective-policy] Iter 51/2000 total_loss=-8.031148e-01 policy_loss=-5.582587e+00 theta_norm=1.897367e+01 beta_norm=1.561470e-01 grad_norm=1.384958e+01 policy_grad_norm=9.486833e+00 state_weight_update=clipped state_weight_min=-1.006989e-03 state_weight_max=9.890862e-02 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS objective-policy] Iter 101/2000 total_loss=-7.088651e-01 policy_loss=-6.225919e+00 theta_norm=1.897367e+01 beta_norm=2.782550e-01 grad_norm=1.143437e+01 policy_grad_norm=9.486833e+00 state_weight_update=clipped state_weight_min=-1.987821e-03 state_weight_max=9.956981e-02 policy_state_weight

Much better clipping!

### Tabular softmax MA

In [20]:
solver_e = VBetaLogitSolver(
    mdp=mdp,
    csv_path=dataset_path,
    beta=1e-2,          # ridge
    seed=SEED,
    device=DEVICE,
)

evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))

In [21]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
)
evaluator_e.print_policy()

[FOGAS logit-policy] Iter 1/2000 total_loss=-9.486833e-01 policy_loss=-4.743416e+00 theta_norm=1.897367e+01 beta_norm=3.395861e-03 grad_norm=1.698100e+01 policy_grad_norm=9.486833e+00 psi_norm=9.486833e-03 state_weight_update=clipped state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS logit-policy] Iter 51/2000 total_loss=-8.031148e-01 policy_loss=-5.582587e+00 theta_norm=1.897367e+01 beta_norm=1.561470e-01 grad_norm=1.384958e+01 policy_grad_norm=9.486833e+00 psi_norm=4.829003e-01 state_weight_update=clipped state_weight_min=-1.006989e-03 state_weight_max=9.890862e-02 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS logit-policy] Iter 101/2000 total_loss=-7.088651e-01 policy_loss=-6.225919e+00 theta_norm=1.897367e+01 beta_norm=2.782550e-01 grad_norm=1.143437e+01 policy_grad_norm=9.486833e+00 psi_norm=9.524642e-01 state_weight_update=clipped state_weight_min=-1.987821

### Tabular SGD/Adam

In [23]:
policy_features = TabularPolicyFeatures(mdp.N, mdp.A)

solver_e = LinearPolicyFOGAS(
    mdp=mdp,
    csv_path=dataset_path,
    seed=SEED,
    device=DEVICE,
    policy_features=policy_features,
)


evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(
    solver=solver_e,
    metric_callable=evaluator_e.get_metric("reward"),
)

In [25]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
)
evaluator_e.print_policy()

[FOGAS linear-policy] Iter 1/2000 total_loss=-9.486833e-01 policy_objective=-4.743416e+00 theta_norm=1.897367e+01 beta_norm=3.661792e-03 grad_norm=1.831079e+01 policy_grad_norm=0.000000e+00 psi_norm=0.000000e+00 policy_optimizer=adam state_weight_update=clipped state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-policy] Iter 51/2000 total_loss=-7.922126e-01 policy_objective=-5.577714e+00 theta_norm=1.897367e+01 beta_norm=1.686164e-01 grad_norm=1.490270e+01 policy_grad_norm=3.031729e-01 psi_norm=3.056484e-01 policy_optimizer=adam state_weight_update=clipped state_weight_min=-1.014779e-03 state_weight_max=9.643782e-02 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-policy] Iter 101/2000 total_loss=-6.911284e-01 policy_objective=-6.202907e+00 theta_norm=1.897367e+01 beta_norm=2.992139e-01 grad_norm=1.192287e+01 policy_grad_norm=4.962606e-01 psi_norm=6.991

Here Adam is much better than SGD and we can notice the accumulation of momentum and we have higuer probabilities in optimal states.

## Beta update

### Linear $u_{\beta}$ adaptation

In [30]:
u_features = TabularPolicyFeatures(N, A)
policy_features = TabularPolicyFeatures(N, A)

solver_e = LinearBetaPiSolver(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    x0=x_0,
    csv_path=dataset_path,
    u_function=LinearUFunction(u_features),
    policy_features=policy_features,
    seed=SEED,
    device=DEVICE,
)

In [31]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
)

[FOGAS linear-beta-pi] Iter 1/2000 total_loss=-9.486833e-01 policy_objective=-4.743416e+00 theta_norm=1.897367e+01 beta_norm=3.640129e-03 grad_norm=1.820247e+01 policy_grad_norm=0.000000e+00 psi_norm=0.000000e+00 policy_optimizer=adam state_weight_update=clipped state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-beta-pi] Iter 51/2000 total_loss=-7.922087e-01 policy_objective=-5.577755e+00 theta_norm=1.897367e+01 beta_norm=1.673958e-01 grad_norm=1.476893e+01 policy_grad_norm=3.031775e-01 psi_norm=3.056773e-01 policy_optimizer=adam state_weight_update=clipped state_weight_min=-1.013934e-03 state_weight_max=9.643587e-02 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-beta-pi] Iter 101/2000 total_loss=-6.911267e-01 policy_objective=-6.202983e+00 theta_norm=1.897367e+01 beta_norm=2.965224e-01 grad_norm=1.175470e+01 policy_grad_norm=4.962780e-01 psi_norm=6.

tensor([[0.1219, 0.4835, 0.1265, 0.2681],
        [0.1912, 0.2211, 0.0875, 0.5003],
        [0.2423, 0.5308, 0.0130, 0.2138],
        [0.0752, 0.6737, 0.1178, 0.1333],
        [0.2500, 0.2500, 0.2500, 0.2500],
        [0.0291, 0.4791, 0.0127, 0.4791],
        [0.0414, 0.1201, 0.1225, 0.7160],
        [0.1583, 0.1640, 0.0437, 0.6339],
        [0.2500, 0.2500, 0.2500, 0.2500]])

In [29]:
mdp.print_policy(solver_e.pi)

  State 0: π(a=0|s=0) = 0.12  π(a=1|s=0) = 0.48  π(a=2|s=0) = 0.13  π(a=3|s=0) = 0.27  --> best action: 1
  State 1: π(a=0|s=1) = 0.19  π(a=1|s=1) = 0.22  π(a=2|s=1) = 0.09  π(a=3|s=1) = 0.50  --> best action: 3
  State 2: π(a=0|s=2) = 0.24  π(a=1|s=2) = 0.53  π(a=2|s=2) = 0.01  π(a=3|s=2) = 0.21  --> best action: 1
  State 3: π(a=0|s=3) = 0.08  π(a=1|s=3) = 0.67  π(a=2|s=3) = 0.12  π(a=3|s=3) = 0.13  --> best action: 1
  State 4: π(a=0|s=4) = 0.25  π(a=1|s=4) = 0.25  π(a=2|s=4) = 0.25  π(a=3|s=4) = 0.25  --> best action: 0
  State 5: π(a=0|s=5) = 0.03  π(a=1|s=5) = 0.48  π(a=2|s=5) = 0.01  π(a=3|s=5) = 0.48  --> best action: 1
  State 6: π(a=0|s=6) = 0.04  π(a=1|s=6) = 0.12  π(a=2|s=6) = 0.12  π(a=3|s=6) = 0.72  --> best action: 3
  State 7: π(a=0|s=7) = 0.16  π(a=1|s=7) = 0.16  π(a=2|s=7) = 0.04  π(a=3|s=7) = 0.63  --> best action: 3
  State 8: π(a=0|s=8) = 0.25  π(a=1|s=8) = 0.25  π(a=2|s=8) = 0.25  π(a=3|s=8) = 0.25  --> best action: 0



## Theta update

### Loss dependency beta regularizer

In [33]:
u_features = TabularPolicyFeatures(N, A)
policy_features = TabularPolicyFeatures(N, A)

solver_e = LossThetaBetaPiSolver(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    x0=x_0,
    csv_path=dataset_path,
    u_function=LinearUFunction(u_features),
    policy_features=policy_features,
    seed=SEED,
    device=DEVICE,
)

In [34]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=False,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
    theta_loss_include_beta_reg=True,
)
mdp.print_policy(solver_e.pi)

  State 0: π(a=0|s=0) = 0.12  π(a=1|s=0) = 0.48  π(a=2|s=0) = 0.13  π(a=3|s=0) = 0.27  --> best action: 1
  State 1: π(a=0|s=1) = 0.19  π(a=1|s=1) = 0.22  π(a=2|s=1) = 0.09  π(a=3|s=1) = 0.50  --> best action: 3
  State 2: π(a=0|s=2) = 0.24  π(a=1|s=2) = 0.53  π(a=2|s=2) = 0.01  π(a=3|s=2) = 0.21  --> best action: 1
  State 3: π(a=0|s=3) = 0.08  π(a=1|s=3) = 0.67  π(a=2|s=3) = 0.12  π(a=3|s=3) = 0.13  --> best action: 1
  State 4: π(a=0|s=4) = 0.25  π(a=1|s=4) = 0.25  π(a=2|s=4) = 0.25  π(a=3|s=4) = 0.25  --> best action: 0
  State 5: π(a=0|s=5) = 0.03  π(a=1|s=5) = 0.48  π(a=2|s=5) = 0.01  π(a=3|s=5) = 0.48  --> best action: 1
  State 6: π(a=0|s=6) = 0.04  π(a=1|s=6) = 0.12  π(a=2|s=6) = 0.12  π(a=3|s=6) = 0.72  --> best action: 3
  State 7: π(a=0|s=7) = 0.16  π(a=1|s=7) = 0.16  π(a=2|s=7) = 0.04  π(a=3|s=7) = 0.63  --> best action: 3
  State 8: π(a=0|s=8) = 0.25  π(a=1|s=8) = 0.25  π(a=2|s=8) = 0.25  π(a=3|s=8) = 0.25  --> best action: 0



In [35]:
solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=False,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
    theta_loss_include_beta_reg=False,
)
mdp.print_policy(solver_e.pi)

  State 0: π(a=0|s=0) = 0.12  π(a=1|s=0) = 0.48  π(a=2|s=0) = 0.13  π(a=3|s=0) = 0.27  --> best action: 1
  State 1: π(a=0|s=1) = 0.19  π(a=1|s=1) = 0.22  π(a=2|s=1) = 0.09  π(a=3|s=1) = 0.50  --> best action: 3
  State 2: π(a=0|s=2) = 0.24  π(a=1|s=2) = 0.53  π(a=2|s=2) = 0.01  π(a=3|s=2) = 0.21  --> best action: 1
  State 3: π(a=0|s=3) = 0.08  π(a=1|s=3) = 0.67  π(a=2|s=3) = 0.12  π(a=3|s=3) = 0.13  --> best action: 1
  State 4: π(a=0|s=4) = 0.25  π(a=1|s=4) = 0.25  π(a=2|s=4) = 0.25  π(a=3|s=4) = 0.25  --> best action: 0
  State 5: π(a=0|s=5) = 0.03  π(a=1|s=5) = 0.48  π(a=2|s=5) = 0.01  π(a=3|s=5) = 0.48  --> best action: 1
  State 6: π(a=0|s=6) = 0.04  π(a=1|s=6) = 0.12  π(a=2|s=6) = 0.12  π(a=3|s=6) = 0.72  --> best action: 3
  State 7: π(a=0|s=7) = 0.16  π(a=1|s=7) = 0.16  π(a=2|s=7) = 0.04  π(a=3|s=7) = 0.63  --> best action: 3
  State 8: π(a=0|s=8) = 0.25  π(a=1|s=8) = 0.25  π(a=2|s=8) = 0.25  π(a=3|s=8) = 0.25  --> best action: 0



No change neither!

### Regularized update

Same of 3 grid, adaptive for sure and adam works much better than SGD, but there is the need of more steps to do the best update.

In [14]:
u_features = TabularPolicyFeatures(N, A)
policy_features = TabularPolicyFeatures(N, A)

solver_e = RegularizedLossThetaBetaPiSolver(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    x0=x_0,
    csv_path=dataset_path,
    u_function=LinearUFunction(u_features),
    policy_features=policy_features,
    seed=SEED,
    device=DEVICE,
)

solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
    theta_loss_include_beta_reg=False,
    theta_update="adam",
    theta_lambda_mode="adaptive",
    theta_lr=1e-2,          
    theta_inner_steps=50, 
)

mdp.print_policy(solver_e.pi)


[FOGAS linear-beta-pi] Iter 1/2000 total_loss=-4.967739e-02 policy_objective=-2.483869e-01 theta_norm=9.935478e-01 beta_norm=4.421997e-04 grad_norm=2.211220e+00 policy_grad_norm=0.000000e+00 psi_norm=0.000000e+00 policy_optimizer=adam state_weight_update=clipped state_weight_min=0.000000e+00 state_weight_max=1.000000e-01 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-beta-pi] Iter 51/2000 total_loss=-4.812170e-02 policy_objective=-2.990328e-01 theta_norm=1.004590e+00 beta_norm=2.222961e-02 grad_norm=2.161038e+00 policy_grad_norm=7.858170e-03 psi_norm=2.485709e-01 policy_optimizer=adam state_weight_update=clipped state_weight_min=-1.014115e-03 state_weight_max=9.978093e-02 policy_state_weight_min=5.000000e-01 policy_state_weight_max=5.000000e-01
[FOGAS linear-beta-pi] Iter 101/2000 total_loss=-4.669793e-02 policy_objective=-3.396650e-01 theta_norm=1.027951e+00 beta_norm=4.349197e-02 grad_norm=2.128029e+00 policy_grad_norm=1.210019e-02 psi_norm=5.

# Linear

In [17]:
u_features = TabularFeatures(N, A)
q_features = TabularFeatures(N, A)
policy_features = TabularFeatures(N, A)

solver_e = LinearSolver(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    x0=x_0,
    csv_path=dataset_path,
    u_function=LinearFunction(u_features),
    q_function=LinearQFunction(q_features),
    policy_features=policy_features,
    seed=SEED,
    device=DEVICE,
    theta_loss_include_beta_reg=False,
    theta_update="adam",
    theta_lr=1e-2,
    theta_inner_steps=100,
    d_theta_scale=1.0,
)

solver_e.run(
    alpha=1e-3,
    eta=2e-4,
    rho=0.5,
    policy_optimizer="adam",
    tqdm_print=False,
    verbose=True,
    log_interval=50,
    T=2000,
    state_weight_update="clipped",
    c_min=0.5,
)

mdp.print_policy(solver_e.pi)

[LinearSolver] Iter 1/2000 iter=0 total_loss=-9.815896e-02 policy_objective=-4.907948e-01 beta_objective=0.000000e+00 q_objective=-9.308077e-02 policy_grad_norm=0.000000e+00 beta_grad_norm=1.362028e-01 theta_grad_norm=4.482657e-02 D_theta=1.897367e+01 effective_D_theta=1.897367e+01
[LinearSolver] Iter 51/2000 iter=50 total_loss=-9.532365e-02 policy_objective=-5.838659e-01 beta_objective=2.835099e-03 q_objective=-9.135781e-02 policy_grad_norm=1.424855e-02 beta_grad_norm=1.252017e-01 theta_grad_norm=4.394702e-02 D_theta=1.897367e+01 effective_D_theta=1.897367e+01
[LinearSolver] Iter 101/2000 iter=100 total_loss=-9.299048e-02 policy_objective=-6.647483e-01 beta_objective=5.167756e-03 q_objective=-8.997573e-02 policy_grad_norm=2.385431e-02 beta_grad_norm=1.167031e-01 theta_grad_norm=4.316305e-02 D_theta=1.897367e+01 effective_D_theta=1.897367e+01
[LinearSolver] Iter 151/2000 iter=150 total_loss=-9.101690e-02 policy_objective=-7.357656e-01 beta_objective=7.140629e-03 q_objective=-8.884026e-

# RBF Features

In [118]:
# -----------------------------
# Problem definition
# -----------------------------
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)
N = len(states)
A = len(actions)
gamma = 0.9

x_0 = 0
dataset_path = str(DATASETS_DIR / "3gridw_rbf.csv")

goal = 8
pit  = 5
wall = 4

step_cost   = -0.01
goal_reward = 1.0
pit_reward  = -1.0

# -----------------------------
# Helpers
# -----------------------------
def to_rc(s):
    return divmod(int(s), 3)

def to_s(r, c):
    return int(r) * 3 + int(c)

def get_norm_coords(s):
    r, c = to_rc(s)
    return torch.tensor([r / 2.0, c / 2.0], dtype=torch.float64)

# -----------------------------
# Deterministic dynamics
# -----------------------------
def next_state(s, a):
    s = int(s)
    a = int(a)

    if s == goal or s == pit:
        return s

    r, c = to_rc(s)

    if a == 0:
        r2, c2 = max(0, r - 1), c      # up
    elif a == 1:
        r2, c2 = min(2, r + 1), c      # down
    elif a == 2:
        r2, c2 = r, max(0, c - 1)      # left
    elif a == 3:
        r2, c2 = r, min(2, c + 1)      # right
    else:
        raise ValueError("Invalid action")

    sp = to_s(r2, c2)
    if sp == wall:
        return s
    return sp

# -----------------------------
# Transition matrix
# -----------------------------
P = torch.zeros((N * A, N), dtype=torch.float64)
for x in states:
    for a in actions:
        idx = int(x) * A + int(a)
        xp = next_state(x, a)
        P[idx, xp] = 1.0

# -----------------------------
# Reward
# -----------------------------
def reward_fn(x, a):
    x = int(x)
    if x == goal:
        return goal_reward
    if x == pit:
        return pit_reward
    return step_cost

# -----------------------------
# RBF features
# phi(x,a) = e_a ⊗ phi_state(x)
# -----------------------------
grid_1d = torch.tensor([0.25, 0.75], dtype=torch.float64)
centers = torch.cartesian_prod(grid_1d, grid_1d)

def calculate_local_sigma(centers, k=2):
    dist_matrix = torch.cdist(centers, centers, p=2)
    topk_dists, _ = torch.topk(dist_matrix, k + 1, largest=False, dim=1)
    nearest_neighbor_dists = topk_dists[:, 1]
    return torch.mean(nearest_neighbor_dists)

rbf_sigma = calculate_local_sigma(centers, k=2) * 4

def phi_state(x):
    coords = get_norm_coords(x)
    dist_sq = torch.sum((coords - centers) ** 2, dim=1)
    rbf = torch.exp(-dist_sq / (2 * rbf_sigma ** 2))

    is_pit = 1.0 if int(x) == pit else 0.0
    is_goal = 1.0 if int(x) == goal else 0.0

    return torch.cat([
        rbf,
        torch.tensor([is_pit, is_goal], dtype=torch.float64),
    ])

def phi(x, a):
    s_feat = phi_state(x)
    e_a = torch.zeros(A, dtype=torch.float64)
    e_a[int(a)] = 1.0
    return torch.kron(e_a, s_feat)

d = int(phi(states[0], actions[0]).shape[0])
print("Feature dimension:", d)

# -----------------------------
# Build MDP / solver input
# -----------------------------
mdp = PolicySolver(
    states=states,
    actions=actions,
    phi=phi,
    reward_fn=reward_fn,
    gamma=gamma,
    x0=x_0,
    P=P,
)

Feature dimension: 24


### Empirical

In [119]:
collector = EnvDataCollector(
    mdp=mdp,
    max_steps=30,
    terminal_states=[goal, pit],
    seed=42,
)

df = collector.collect_mixed_dataset_terminal_aware(
    policies=[
        (mdp.pi_star, 0.4),   # epsilon-greedy around pi_star
        "random",
    ],
    proportions=[0.8, 0.2],
    n_steps=1000,
    save_path=dataset_path,
    verbose=True,
    episode_based=True,
    extra_steps=2,
)


  MIXED TERMINAL-AWARE DATASET COLLECTION SUMMARY (TORCH)
Total transitions: 1000
Total episodes: 106
Extra steps: 2

Policy Distribution:
  Policy 0:   791 steps (79.1%) | Target: 80.0% | Episodes: 91
  Policy 1:   209 steps (20.9%) | Target: 20.0% | Episodes: 16

✅ Mixed terminal-aware dataset saved to: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/3gridw_rbf.csv


In [120]:
solver_e = FOGASSolverBetaVectorized(mdp=mdp, print_params=True, csv_path=dataset_path, seed=SEED, device=DEVICE)
evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))


MDP omega not provided. Estimating from dataset...

Device: cpu
Dataset: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/3gridw_rbf.csv (n=1000)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           1000
Feature norm bound R:     2.1562
Num states N:             9
Num actions A:            4
Feature dim d:            24
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      4302.854671800086
T (iterations):                4303

FOGAS Hyperparameters
---------------------
alpha:                        0.000240
rho:                            424.257383
eta:                            0.000006
D_theta:                    15.491933
beta (ridge):             0.000045
D_pi (derived):           16.019217


Estimated omega (first 5 components): tensor([-6.7756e-03, -8.1840e-03,  2.0385e-04,  4.1750e-03, -9.7534e-01])


In [ ]:
solver_e.run(alpha=1e-2, eta=2e-4, rho=0.5, tqdm_print=True, T = 4000)
evaluator_e.compare_value_functions()
evaluator_e.compare_final_rewards()
evaluator_e.print_policy()

FOGAS:   0%|          | 0/4000 [00:00<?, ?it/s]

FOGAS: 100%|██████████| 4000/4000 [00:01<00:00, 2368.62it/s]


========== VALUE FUNCTION COMPARISON ==========

State-wise V comparison:
State 0: V*(x) =  6.526610 | V^π(x) =  2.589096 | Δ = -3.937514e+00
State 1: V*(x) =  5.863949 | V^π(x) =  0.382445 | Δ = -5.481504e+00
State 2: V*(x) =  5.267554 | V^π(x) = -2.756923 | Δ = -8.024477e+00
State 3: V*(x) =  7.262900 | V^π(x) =  3.872153 | Δ = -3.390747e+00
State 4: V*(x) =  8.081000 | V^π(x) = -0.301057 | Δ = -8.382057e+00
State 5: V*(x) = -10.000000 | V^π(x) = -10.000000 | Δ =  1.243450e-14
State 6: V*(x) =  8.081000 | V^π(x) =  5.482169 | Δ = -2.598831e+00
State 7: V*(x) =  8.990000 | V^π(x) =  7.566916 | Δ = -1.423084e+00
State 8: V*(x) =  10.000000 | V^π(x) =  10.000000 | Δ = -1.243450e-14

Action-value Q comparison:
(x=0, a=0): Q*(x,a) =  5.863949 | Q^π(x,a) =  2.320187 | Δ = -3.543762e+00
(x=0, a=1): Q*(x,a) =  6.526610 | Q^π(x,a) =  3.474938 | Δ = -3.051672e+00
(x=0, a=2): Q*(x,a) =  5.863949 | Q^π(x,a) =  2.320187 | Δ = -3.543762e+00
(x=0, a=3): Q*(x,a) =  5.267554 | Q^π(x,a) =  0.334201 |